### 线性回归的简洁实现

#### 生成数据集

In [1]:
# import numpy as np
import torch
# from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

#### 读取数据集

In [2]:
# def load_array(data_arrays, batch_size, is_train=True):  #@save
#     """构造一个PyTorch数据迭代器"""
#     dataset = data.TensorDataset(*data_arrays)
#     return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [3]:
batch_size = 10
data_iter = d2l.load_array((features, labels), batch_size)

In [4]:
next(iter(data_iter))   # 读取并打印第一个小批量样本，使用iter构造Python迭代器，并使用next从迭代器中获取第一项

[tensor([[ 0.8109,  0.1446],
         [-1.7492, -1.0784],
         [ 0.2756,  0.1401],
         [ 0.0907, -0.6122],
         [-1.2601, -0.7901],
         [-0.8516,  0.0278],
         [-0.9820, -0.2220],
         [-3.0957, -1.6534],
         [ 0.1229,  0.4239],
         [ 0.2539, -1.4203]]),
 tensor([[5.3279],
         [4.3745],
         [4.2587],
         [6.4682],
         [4.3614],
         [2.3993],
         [2.9868],
         [3.6342],
         [3.0074],
         [9.5307]])]

#### 定义模型

In [5]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1)) # 首先定义一个模型变量net，它是一个Sequential类的实例。 

#### 初始化模型参数

In [6]:
# 指定每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样， 偏置参数将初始化为零。
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

#### 定义损失函数

In [7]:
loss = nn.MSELoss() # 计算均方误差使用的是MSELoss类，也称为平方L2范数。 默认情况下，它返回所有样本损失的平均值。

#### 定义优化算法

In [8]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

#### 训练

In [9]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y) # 计算损失
        trainer.zero_grad() # 清空梯度
        l.backward()        # 反向传播
        trainer.step()      # 更新参数
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000209
epoch 2, loss 0.000102
epoch 3, loss 0.000101


In [10]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([-0.0002, -0.0002])
b的估计误差： tensor([0.0008])


### 练习

#### 1. 如果将小批量的总损失替换为小批量损失的平均值，需要如何更改学习率？

In [11]:
# loss_fn = nn.MSELoss(reduction="mean")    # defualt param
# optimizer = torch.optim.SGD(model.parameters(), lr=0.03)

如果原代码是总损失，即 reduction="sum"，lr=0.03，改为 reduction="mean"，则 lr 改为 0.03 * batch_size

#### 2. 查看深度学习框架文档，它们提供了哪些损失函数和初始化方法？用Huber损失代替原损失，即
<math xmlns="http://www.w3.org/1998/Math/MathML" display="block">
  <mtable displaystyle="true" columnalign="right" columnspacing="0em" rowspacing="3pt">
    <mtr>
      <mtd>
        <mi>l</mi>
        <mo stretchy="false">(</mo>
        <mi>y</mi>
        <mo>,</mo>
        <msup>
          <mi>y</mi>
          <mo data-mjx-alternate="1">&#x2032;</mo>
        </msup>
        <mo stretchy="false">)</mo>
        <mo>=</mo>
        <mrow data-mjx-texclass="INNER">
          <mo data-mjx-texclass="OPEN">{</mo>
          <mtable columnalign="left left" columnspacing="1em" rowspacing=".2em">
            <mtr>
              <mtd>
                <mo stretchy="false">|</mo>
                <mi>y</mi>
                <mo>&#x2212;</mo>
                <msup>
                  <mi>y</mi>
                  <mo data-mjx-alternate="1">&#x2032;</mo>
                </msup>
                <mrow data-mjx-texclass="ORD">
                  <mo stretchy="false">|</mo>
                </mrow>
                <mo>&#x2212;</mo>
                <mfrac>
                  <mi>&#x3C3;</mi>
                  <mn>2</mn>
                </mfrac>
              </mtd>
              <mtd>
                <mtext>&#xA0;if&#xA0;</mtext>
                <mo stretchy="false">|</mo>
                <mi>y</mi>
                <mo>&#x2212;</mo>
                <msup>
                  <mi>y</mi>
                  <mo data-mjx-alternate="1">&#x2032;</mo>
                </msup>
                <mrow data-mjx-texclass="ORD">
                  <mo stretchy="false">|</mo>
                </mrow>
                <mo>&gt;</mo>
                <mi>&#x3C3;</mi>
              </mtd>
            </mtr>
            <mtr>
              <mtd>
                <mfrac>
                  <mn>1</mn>
                  <mrow>
                    <mn>2</mn>
                    <mi>&#x3C3;</mi>
                  </mrow>
                </mfrac>
                <mo stretchy="false">(</mo>
                <mi>y</mi>
                <mo>&#x2212;</mo>
                <msup>
                  <mi>y</mi>
                  <mo data-mjx-alternate="1">&#x2032;</mo>
                </msup>
                <msup>
                  <mo stretchy="false">)</mo>
                  <mn>2</mn>
                </msup>
              </mtd>
              <mtd>
                <mtext>&#xA0;&#x5176;&#x5B83;&#x60C5;&#x51B5;</mtext>
              </mtd>
            </mtr>
          </mtable>
          <mo data-mjx-texclass="CLOSE" fence="true" stretchy="true" symmetric="true"></mo>
        </mrow>
      </mtd>
    </mtr>
  </mtable>
</math>

常见损失函数包括：

- 回归：`MSELoss`、`L1Loss`、`SmoothL1Loss`、`HuberLoss`
- 分类：`CrossEntropyLoss`、`NLLLoss`
- 二分类：`BCELoss`、`BCEWithLogitsLoss`
- 其他：`KLDivLoss`、`PoissonNLLLoss`、`CTCLoss`、`TripletMarginLoss` 等

常见初始化方法包括：
- `normal_`：正态分布初始化
- `uniform_`：均匀分布初始化
- `zeros_`、`ones_`、`constant_`：常数初始化
- `xavier_uniform_`、`xavier_normal_`：Xavier 初始化
- `kaiming_uniform_`、`kaiming_normal_`：He/Kaiming 初始化
- `orthogonal_`：正交初始化
- `trunc_normal_`：截断正态分布初始化

In [12]:
sigma = 1.0
loss = nn.SmoothL1Loss(beta=sigma, reduction="mean")

In [13]:
num_epochs = 3

for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)

        trainer.zero_grad()
        l.backward()
        trainer.step()

    l = loss(net(features), labels)
    print(f"epoch {epoch + 1}, loss {l:f}")

epoch 1, loss 0.000050
epoch 2, loss 0.000050
epoch 3, loss 0.000050


#### 3. 如何访问线性回归的梯度？

In [14]:
print("权重梯度：", net[0].weight.grad)
print("偏置梯度：", net[0].bias.grad)

权重梯度： tensor([[-0.0003,  0.0036]])
偏置梯度： tensor([-0.0019])
